# Topsis-for-pre-trained-models Assignment

In [5]:
from datasets import load_dataset
import time
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

In [6]:
dataset = load_dataset("SetFit/sst2")
test_data = dataset["test"]

TEXT_COL = "text"
LABEL_COL = "label"

MAX_SAMPLES = 200

Repo card metadata block was not found. Setting CardData to empty.


In [7]:
def evaluate_model(model_name, dataset):
    """
    Evaluates a pretrained Hugging Face text classification model
    and returns Accuracy, F1-score, Inference Time, and Model Size.
    """

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False
    )
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()

    texts = dataset[TEXT_COL][:MAX_SAMPLES]
    labels = dataset[LABEL_COL][:MAX_SAMPLES]

    predictions = []
    start_time = time.time()

    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                padding=True
            )
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            predictions.append(pred)

    end_time = time.time()

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    inference_time = end_time - start_time
    model_size = sum(p.numel() for p in model.parameters()) / 1e6  # MB approx

    return accuracy, f1, inference_time, model_size

In [8]:
models = [
    "distilbert-base-uncased-finetuned-sst-2-english",
    "bert-base-uncased",
    "roberta-base",
    "albert-base-v2",
    "xlnet-base-cased"
]

In [9]:
results = []

for model in models:
    print(f"Evaluating: {model}")
    acc, f1, time_taken, size = evaluate_model(model, test_data)
    results.append([model, acc, f1, time_taken, size])

Evaluating: distilbert-base-uncased-finetuned-sst-2-english


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Evaluating: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.bias                    | UNEXPECTED | 
lm_loss.weight                  | UNEXPECTED | 
sequence_summary.summary.bias   | MISSING    | 
logits_proj.bias                | MISSING    | 
sequence_summary.summary.weight | MISSING    | 
logits_proj.weight              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "F1 Score", "Inference Time", "Model Size (MB)"]
)

decision_matrix = df[["Accuracy", "F1 Score", "Inference Time", "Model Size (MB)"]].values

In [11]:
def topsis(matrix, weights, impacts):
    #
    norm_matrix = matrix / np.sqrt((matrix ** 2).sum(axis=0))

    weighted_matrix = norm_matrix * weights

    ideal_best = np.zeros(matrix.shape[1])
    ideal_worst = np.zeros(matrix.shape[1])

    for i in range(len(impacts)):
        if impacts[i] == "+":
            ideal_best[i] = weighted_matrix[:, i].max()
            ideal_worst[i] = weighted_matrix[:, i].min()
        else:
            ideal_best[i] = weighted_matrix[:, i].min()
            ideal_worst[i] = weighted_matrix[:, i].max()

    dist_best = np.sqrt(((weighted_matrix - ideal_best) ** 2).sum(axis=1))
    dist_worst = np.sqrt(((weighted_matrix - ideal_worst) ** 2).sum(axis=1))

    scores = dist_worst / (dist_best + dist_worst)
    return scores

In [12]:
weights = np.array([0.25, 0.25, 0.25, 0.25])
impacts = ["+", "+", "-", "-"]

df["TOPSIS Score"] = topsis(decision_matrix, weights, impacts)
df["Rank"] = df["TOPSIS Score"].rank(ascending=False)

df = df.sort_values("Rank")
print(df)

                                             Model  Accuracy  F1 Score  \
0  distilbert-base-uncased-finetuned-sst-2-english     0.925  0.929577   
3                                   albert-base-v2     0.520  0.684211   
2                                     roberta-base     0.520  0.684211   
1                                bert-base-uncased     0.565  0.497110   
4                                 xlnet-base-cased     0.490  0.150000   

   Inference Time  Model Size (MB)  TOPSIS Score  Rank  
0       19.291003        66.955010      0.750095   1.0  
3       29.094010        11.685122      0.657833   2.0  
2       32.967459       124.647170      0.392564   3.0  
1       38.268237       109.483778      0.305508   4.0  
4       48.132757       117.310466      0.037557   5.0  


In [13]:
df.to_csv("topsis_model_ranking.csv", index=False)